In [11]:
import pandas as pd
import os
import sys
# Aggiunge la cartella src al path
sys.path.append(os.path.abspath('../src'))
from transformers import AutoTokenizer, AutoModel
import torch

In [12]:
#df = pd.read_csv('../data/nucleosome_dataset_GSE36979_hg19_whole_genome_201pb_balanced.csv')
#df = pd.read_csv('../data/dataset_RestingNucleosomes_chr20.csv')
#df = pd.read_csv('../data/cd4t_data/CD4T_h19_Act_tot.csv')
#df = pd.read_csv('../data/cd4t_data/CD4T_h19_Act_tot.csv')
#df = pd.read_csv('../data/cd4t_data/CD4T_h19_Act_tot_99_8_percentile.csv')
df = pd.read_csv('../data/lymphoblasotid_data/Lymphoblastoid_99_8_percentile.csv')
#df = pd.read_csv('../data/cd4t_data/CD4T_h19_Rest_tot_99_8_percentile.csv')

#df = pd.read_csv('../data/cd4t_data/CD4T_h19_Rest_tot_99_9.csv')

#df = pd.read_csv('../data/dataframe_nup_2/df_nup2_human_prom.csv')
#df = pd.read_csv('../data/dataframe_nup_1/df_nup1_sapiens.csv')
df['nucleosoma'].value_counts()

#df = df[df['chr']=='chr2']

nucleosoma
0    125255
1     97358
Name: count, dtype: int64

In [3]:
df = df.rename(columns={'sequenza':'sequence', 'nucleosoma':'label'})

#se voglio REVERSE ---------
#df['sequence']= df.apply(lambda x: x['sequence'][::-1], axis=1)
#---------------


df.head()

,sequence,label,chr
0,GAGCGGATGGTACTTCATTCATGCCTTTGACACATAAAGCCTGGAA...,1,chr6
1,TTGACAATCCTTCCTATAGCATTTATATGCCAAATAAATAAGCCAT...,0,chr11
2,TGTGATGTGTGAATTCAACTCACAGTGTGGATCCTTCCTTTTGATA...,1,chr6
3,AACACTAGGCAAAGATCACAAAATTATGATTTATTGTTTCACTGTC...,0,chr4
4,CCTTTACCTCTGTTGTCAGGGCCACAGGGAAACCCGCACCCAGCCG...,1,chr3


In [4]:
import pandas as pd

# Supponiamo che il tuo dataframe si chiami 'df'

# 1. Separa le righe con label 1 e label 0
df_ones = df[df['label'] == 1]
df_zeros = df[df['label'] == 0]

# 2. Esegui il campionamento casuale (sample) sugli 0
# n=len(df_ones) dice di prenderne tanti quanti sono gli 1
# random_state serve a rendere l'estrazione riproducibile (opzionale)
df_zeros_sampled = df_zeros.sample(n=len(df_ones), random_state=42)

# 3. Unisci (concat) i due dataframe: tutti gli 1 e gli 0 scelti
df_balanced = pd.concat([df_ones, df_zeros_sampled])

# 4. (Opzionale) Mischia le righe del risultato finale in modo che 
# 0 e 1 non siano ordinati
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Verifica del risultato
print(df_balanced['label'].value_counts())

label
1    58489
0    58489
Name: count, dtype: int64


In [5]:
df = df_balanced

In [6]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import pandas as pd

# # Imposta il dispositivo
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# # Carica modello e tokenizer una sola volta
# tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/nucleotide-transformer-2.5B-multi-species")
# model = AutoModel.from_pretrained(
#     "InstaDeepAI/nucleotide-transformer-2.5B-multi-species",
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
# ).to(device)
# model.eval()

# # Funzione per calcolare gli embedding a batch
# def batch_embeddings(sequences, tokenizer, model, device, batch_size=8):
#     all_embeddings = []
#     for i in tqdm(range(0, len(sequences), batch_size)):
#         batch = sequences[i:i+batch_size]
#         tokens = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
#         with torch.no_grad():
#             outputs = model(**tokens)
#             # Usa mean pooling per ottenere un vettore per sequenza
#             batch_emb = outputs.last_hidden_state.cpu()
#         all_embeddings.extend(batch_emb)
#     return all_embeddings

# # Esempio: prendi metà dataset
# #df = df[20000:30000]
# #df = df.sample(frac=0.3, random_state=42).reset_index(drop=True)
# print(df.shape)

# Calcolo in batch
seqs = df["sequence"].str.upper().tolist()
#emb_list = batch_embeddings(seqs, tokenizer, model, device, batch_size=8)

# Aggiungi la colonna degli embedding
df["embedding"] = 0#emb_list


In [7]:
df = df[['sequence', 'label', 'embedding']].to_dict(orient='records')


In [8]:
import pickle
with open('../data/Geo_dataset/CD4T_h19_Rest_tot_99_8_percentile.pkl', 'wb') as f:
    pickle.dump(df, f)

In [9]:
len(df)

116978

In [10]:
assert False

AssertionError: 

In [ ]:
#FINEEEE

In [ ]:
len(pd.read_pickle('../CORENup-A-Combination-of-Convolutional-and-Recurrent-Deep-Neural-Networks-for-NucleosomePositioning/pkl_dir/encoded_nuc_h.pickle')[0])

In [ ]:
df_diff = pd.read_pickle('../data/Geo_dataset/ActivatedNucleosomes_chr1.pkl')

In [ ]:
y_true = [i['label'] for x,i in enumerate(df_diff) if x < 201]
len(y_true)

In [ ]:
with open("output.fasta", "w") as f:
    for i, seq in enumerate(df_diff, start=1):
        name = f"seq{i}"
        print(f">{name}\n{seq['sequence']}\n")
        if i > 200:
            break
        #f.write(f">{name}\n{seq}\n")

In [ ]:
from sklearn.metrics import roc_auc_score, matthews_corrcoef

roc_auc_score(y_true,pd.read_csv('../data/200_seq_NuPoSe', header =None)[1].values)

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time

# Tenta di importare la libreria veloce per la distanza di Levenshtein
# Se non c'è, usa una versione Python standard (più lenta)
try:
    import Levenshtein
    def get_distance(s1, s2):
        return Levenshtein.distance(s1, s2)
except ImportError:
    print("⚠ Libreria 'python-Levenshtein' non trovata. Uso implementazione lenta.")
    print("Per velocità, installala con: pip install python-Levenshtein")
    def get_distance(s1, s2):
        if len(s1) < len(s2):
            return get_distance(s2, s1)
        if len(s2) == 0:
            return len(s1)
        previous_row = range(len(s2) + 1)
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        return previous_row[-1]

##############################################################################
# 1. FUNZIONE DI ANALISI SIMILARITÀ (CAMPIONAMENTO)
##############################################################################

def analyze_similarity_distribution(sequences, num_samples=5000):
    """
    Calcola la similarità su un campione casuale di coppie per capire la
    distribuzione senza dover calcolare la matrice NxN completa.
    """
    print(f"Analisi distribuzione similarità (campionamento {num_samples} coppie)...")
    
    n = len(sequences)
    if n < 2:
        return
    
    identities = []
    seq_len = len(sequences[0]) # Assumiamo tutte 201 bp
    
    # Genera indici casuali
    for _ in range(num_samples):
        i, j = random.sample(range(n), 2)
        s1 = sequences[i]
        s2 = sequences[j]
        
        dist = get_distance(s1, s2)
        
        # Calcolo identità normalizzata (1.0 = identiche, 0.0 = diverse)
        identity = 1.0 - (dist / seq_len)
        identities.append(identity)
        
    # Plotting
    plt.figure(figsize=(10, 6))
    sns.histplot(identities, bins=50, kde=True, color='skyblue', edgecolor='black')
    plt.axvline(x=np.mean(identities), color='red', linestyle='--', label=f'Media: {np.mean(identities):.2f}')
    plt.title('Distribuzione Similarità a Coppie (Campione)')
    plt.xlabel('Identità di Sequenza (0.0 - 1.0)')
    plt.ylabel('Frequenza')
    plt.legend()
    plt.grid(axis='y', alpha=0.5)
    plt.show()
    
    return identities

##############################################################################
# 2. FUNZIONE DI FILTRAGGIO (GREEDY CLUSTERING)
##############################################################################

def remove_highly_similar_sequences(sequences, threshold=0.85):
    """
    Rimuove sequenze che hanno una similarità > threshold rispetto alle 
    sequenze già accettate (Approccio Greedy / Leader Algorithm).
    """
    print(f"\nAvvio rimozione ridondanza (Soglia similarità: {threshold*100:.1f}%)")
    print(f"Input iniziale: {len(sequences)} sequenze")
    
    start_time = time.time()
    
    # Ordiniamo per complessità o casualmente?
    # Spesso si ordina per lunghezza, ma qui sono tutte 201bp.
    # Mescoliamo per evitare bias posizionali se i dati sono ordinati per chr.
    shuffled_seqs = sequences.copy()
    random.shuffle(shuffled_seqs)
    
    unique_sequences = [] 
    seq_len = len(sequences[0])
    
    # Per monitorare il progresso
    total = len(sequences)
    check_interval = max(1, total // 10)
    
    for idx, candidate in enumerate(shuffled_seqs):
        if idx % check_interval == 0:
            print(f"  ...processate {idx}/{total} sequenze. Tenute finora: {len(unique_sequences)}")
            
        is_redundant = False
        
        # Confronta il candidato con le sequenze uniche già trovate (i "centroidi")
        # Nota: Questo diventa lento se unique_sequences cresce molto (>10k).
        for representative in unique_sequences:
            dist = get_distance(candidate, representative)
            identity = 1.0 - (dist / seq_len)
            
            if identity >= threshold:
                is_redundant = True
                break # Trovato un simile, scarta e passa al prossimo
        
        if not is_redundant:
            unique_sequences.append(candidate)
            
    elapsed = time.time() - start_time
    removed = total - len(unique_sequences)
    print(f"\n✓ Filtraggio completato in {elapsed:.2f} secondi.")
    print(f"Sequenze rimosse: {removed} ({(removed/total)*100:.1f}%)")
    print(f"Sequenze rimaste: {len(unique_sequences)}")
    
    return unique_sequences

##############################################################################
# ESEMPIO DI UTILIZZO
##############################################################################

# 1. Creiamo o carichiamo i dati (sostituisci 'dummy_data' con il tuo df['sequenza'].tolist())
# Qui genero dati finti per testare
def generate_dummy_dna(n=1000, length=201):
    bases = ['A', 'C', 'G', 'T']
    seqs = []
    # Creo un set base e poi genero alcune varianti molto simili
    base_templates = [''.join(random.choices(bases, k=length)) for _ in range(n // 10)]
    
    for _ in range(n):
        template = random.choice(base_templates)
        # Introduco mutazioni (1-5 basi) per simulare similarità alta
        s_list = list(template)
        for _ in range(random.randint(0, 20)): # Alcune saranno identiche, altre diverse
            pos = random.randint(0, length-1)
            s_list[pos] = random.choice(bases)
        seqs.append("".join(s_list))
    return seqs

# --- USARE QUESTO BLOCCO ---
# Sostituisci con: my_sequences = df_final['sequenza'].tolist()
print("Generazione dati di test...")
my_sequences = [df_diff[i]['sequence'] for i in range(len(df_diff))]

# A. Analizza
_ = analyze_similarity_distribution(my_sequences)

# B. Filtra (es. teniamo solo sequenze che non sono più simili dell'85%)
clean_sequences = remove_highly_similar_sequences(my_sequences, threshold=0.85)